In [3]:
import numpy as np
import pandas as pd
from scipy.stats import chi2_contingency

# -----------------------
# Helpers
# -----------------------

def chi2_test(name, counts_a, counts_b, labels):
    """
    Run chi-square test for two categorical distributions.
    Returns a dict with stats and effect size.
    """
    table = np.vstack([counts_a, counts_b])
    chi2, p, dof, expected = chi2_contingency(table)

    n = table.sum()
    r, k = table.shape
    cramers_v = np.sqrt(chi2 / (n * (min(r - 1, k - 1))))

    return {
        "test": name,
        "chi2": chi2,
        "dof": dof,
        "p_value": p,
        "cramers_v": cramers_v,
        "labels": labels
    }


def benjamini_hochberg(pvals, alpha=0.05):
    """
    Benjamini–Hochberg FDR correction.
    Returns boolean array of rejections.
    """
    pvals = np.array(pvals)
    m = len(pvals)
    order = np.argsort(pvals)
    ranked = pvals[order]
    thresh = alpha * (np.arange(1, m + 1) / m)

    passed = ranked <= thresh
    reject = np.zeros_like(pvals, dtype=bool)

    if passed.any():
        kmax = np.where(passed)[0].max()
        reject[order[:kmax + 1]] = True

    return reject


# -----------------------
# Input data (FROM TABLES)
# -----------------------

tests = []

# ---- Table 1: Review presence (R_cap) ----
tests.append(
    chi2_test(
        name="T1 Review presence (AI vs Human, R_cap)",
        counts_a=[2781, 6835],   # AI agent: [no review, with review]
        counts_b=[1924, 3650],   # Human
        labels=["no_review", "with_review"]
    )
)

# ---- Table 1: Reviewer composition (reviewed PRs, R_cap) ----
tests.append(
    chi2_test(
        name="T1 Reviewer composition (AI vs Human, R_cap)",
        counts_a=[3939, 552, 2344],  # AI agent: [AI-only, Human-only, Mixed]
        counts_b=[1932, 920, 798],   # Human
        labels=["AI_only", "Human_only", "Mixed"]
    )
)

# ---- Table 2: Review authorship (comments, R_cap) ----
tests.append(
    chi2_test(
        name="T2 Review authorship (AI vs Human, R_cap)",
        counts_a=[16727, 6354],  # AI agent PRs: [C_AI, C_human]
        counts_b=[5589, 4412],   # Human PRs
        labels=["agentic", "human"]
    )
)

# ---- Table 2: Human review categories (comments, R_cap) ----
tests.append(
    chi2_test(
        name="T2 Human review categories (AI vs Human, R_cap)",
        counts_a=[1647, 543, 4164],  # AI agent PRs
        counts_b=[72, 212, 4128],   # Human PRs
        labels=["agentic", "bot", "other"]
    )
)

# -----------------------
# Multiple testing correction
# -----------------------

pvals = [t["p_value"] for t in tests]
reject = benjamini_hochberg(pvals, alpha=0.05)

# -----------------------
# Output
# -----------------------

df = pd.DataFrame(tests)
df["fdr_reject_0.05"] = reject

pd.set_option("display.float_format", "{:.4g}".format)
print(df[["test", "chi2", "dof", "p_value", "cramers_v", "fdr_reject_0.05"]])

# Optional: save to CSV
df.to_csv("chi2_results.csv", index=False)
display(df)

                                              test  chi2  dof    p_value  \
0          T1 Review presence (AI vs Human, R_cap) 51.44    1  7.393e-13   
1     T1 Reviewer composition (AI vs Human, R_cap) 629.4    2 2.161e-137   
2        T2 Review authorship (AI vs Human, R_cap) 873.7    1  5.18e-192   
3  T2 Human review categories (AI vs Human, R_cap)  1280    2  1.33e-278   

   cramers_v  fdr_reject_0.05  
0    0.05819             True  
1      0.245             True  
2     0.1625             True  
3     0.3448             True  


,test,chi2,dof,p_value,cramers_v,labels,fdr_reject_0.05
0,"T1 Review presence (AI vs Human, R_cap)",51.44,1,7.393e-13,0.05819,"[no_review, with_review]",True
1,"T1 Reviewer composition (AI vs Human, R_cap)",629.4,2,2.161e-137,0.245,"[AI_only, Human_only, Mixed]",True
2,"T2 Review authorship (AI vs Human, R_cap)",873.7,1,5.18e-192,0.1625,"[agentic, human]",True
3,"T2 Human review categories (AI vs Human, R_cap)",1280,2,1.33e-278,0.3448,"[agentic, bot, other]",True
